# Praktikum Machine Learning — Klasifikasi Pola Getaran Kompresor

**Sensor:** ESP32 + MPU6050 (sumbu X, Y, Z)  
**Model:** Random Forest  
**Kelas:** `normal` dan `anomali` (*perubahan pola getaran akibat kondisi tumpuan*, bukan diagnosis kerusakan)

**Versi revisi:** memakai **10 fitur** sederhana (RMS, peak-to-peak, frekuensi dominan untuk X/Y/Z, dan RMS gabungan XYZ).

**Tujuan pembelajaran:** mahasiswa dapat menjelaskan urutan **data mentah → visualisasi → window → fitur → pembagian data → training → evaluasi**.

**Cara menggunakan:** Jalankan setiap cell secara berurutan dengan **Shift + Enter**. Baca petunjuk di atas cell sebelum menjalankannya.

**Format ZIP yang diharapkan:** `dataset_kompresor.zip` berisi `normal/normal_01.csv` s.d. `normal_15.csv` dan `anomali/anomali_01.csv` s.d. `anomali_15.csv`. File boleh berada dalam folder tambahan atau dalam satu folder jika namanya memuat `normal` / `anomali`. Kolom wajib CSV: `ax_raw`, `ay_raw`, `az_raw`; kolom `t_us` dianjurkan.

## Tahap 1 — Impor library dan menentukan parameter

- **Sampling 200 Hz** artinya targetnya 200 pengukuran tiap detik.
- **Window 200 sampel** berarti satu potongan sinyal sekitar 1 detik.
- **8192** adalah faktor konversi *raw* menjadi satuan *g* untuk MPU6050 pada rentang **±4g**.

> Jika pengaturan accelerometer di ESP32 berbeda dari ±4g, faktor konversi harus diubah. Frekuensi 200 Hz adalah **target**; kita akan memeriksa interval waktu aktual pada tahap berikutnya.

In [ ]:
# Library untuk membaca ZIP, mengolah angka, membuat plot, dan klasifikasi
import io
import zipfile
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from google.colab import files
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

# Parameter utama praktikum
FS = 200                  # Target sampling ESP32 (Hz)
WINDOW = 200              # Panjang window (sampel)
SKALA = 8192.0            # MPU6050 +/-4g: 8192 raw = 1 g
SEED = 42                 # Agar pembagian acak dapat diulang
KELAS = ["normal", "anomali"]
SUMBU = ["ax_raw", "ay_raw", "az_raw"]

print(f"1 window = {WINDOW / FS:.1f} detik (berdasarkan target sampling)")

## Tahap 2 — Unggah `dataset_kompresor.zip`

Klik **Choose Files**, lalu pilih file ZIP dari laptop. Anda tidak perlu mengunggah CSV satu per satu.

In [ ]:
# Colab meminta Anda memilih ZIP dari laptop
unggahan = files.upload()

# Cari file ZIP yang baru diunggah
nama_zip = next((n for n in unggahan if n.lower().endswith(".zip")), None)
if nama_zip is None:
    raise ValueError("Unggah sebuah file ZIP, misalnya dataset_kompresor.zip")

print("ZIP berhasil dipilih:", nama_zip)

## Tahap 3 — Membaca semua CSV dan memeriksa kualitas data

Kita periksa **jumlah file per kondisi**, jumlah sampel, data hilang, dan **frekuensi sampling aktual** dari `t_us` (jika tersedia).

**Catatan:** data invalid dibuang. Bila banyak sampel hilang atau interval waktu tidak stabil, perbaiki perekaman sebelum percaya pada analisis frekuensi (FFT).

In [ ]:
# Membaca dataset langsung dari ZIP tanpa ekstraksi manual
rekaman = {}       # Berisi data mentah untuk setiap file
ringkasan = []     # Informasi kualitas setiap file

def cari_label(nama):
    # Dahulukan nama file; jika perlu cari nama folder terdekat
    path = Path(nama)
    kandidat = [path.stem.lower()] + [p.lower() for p in reversed(path.parts[:-1])]
    for teks in kandidat:
        if teks.startswith(("anomali", "abnormal", "fault", "modified_support")):
            return "anomali"
        if teks.startswith("normal"):
            return "normal"
    return None

with zipfile.ZipFile(io.BytesIO(unggahan[nama_zip])) as z:
    daftar_csv = [n for n in z.namelist()
                  if n.lower().endswith(".csv") and "__macosx" not in n.lower()]

    for nama in sorted(daftar_csv):
        label = cari_label(nama)
        if label is None:
            print("Dilewati (nama tidak menunjukkan kelas):", nama)
            continue

        df = pd.read_csv(z.open(nama))
        df.columns = df.columns.str.strip().str.lower()
        if not set(SUMBU).issubset(df.columns):
            print("Dilewati (kolom X/Y/Z tidak lengkap):", nama)
            continue

        # Ubah pembacaan sensor menjadi numerik; teks/kosong menjadi NaN
        sensor = df[SUMBU].apply(pd.to_numeric, errors="coerce")
        if "t_us" in df.columns:
            waktu = pd.to_numeric(df["t_us"], errors="coerce")
        else:
            # Waktu perkiraan bila CSV lama tidak memiliki timestamp
            waktu = pd.Series(np.arange(len(df)) * 1e6 / FS, index=df.index)

        baik = sensor.notna().all(axis=1) & waktu.notna()
        invalid = int((~baik).sum())
        bersih = sensor.loc[baik].copy().reset_index(drop=True)
        bersih["t_us"] = waktu.loc[baik].to_numpy()

        if len(bersih) < WINDOW:
            print("Dilewati (< 1 window):", nama)
            continue

        # Periksa selisih waktu antar-sampel (microsecond)
        dt = np.diff(bersih["t_us"].to_numpy())
        dt_positif = dt[dt > 0]
        fs_aktual = 1e6 / dt_positif.mean() if len(dt_positif) else np.nan

        rekaman[nama] = {"label": label, "data": bersih}
        ringkasan.append({"file": nama, "label": label,
                          "sampel": len(bersih), "invalid": invalid,
                          "fs_aktual": fs_aktual})

info = pd.DataFrame(ringkasan)
if info.empty or set(info["label"]) != set(KELAS):
    raise ValueError("Diperlukan CSV valid dari KEDUA kelas: normal dan anomali.")

print("Jumlah file per kelas:")
print(info.groupby("label")["file"].count().to_string())
print("\nRingkasan 5 file pertama:")
display(info.head())

# Praktikum dirancang untuk 15 file di setiap kondisi
for kelas in KELAS:
    n = (info["label"] == kelas).sum()
    if n != 15:
        print(f"PERHATIAN: kelas {kelas} berisi {n} file (rencana: 15 file).")

if info["invalid"].sum() > 0:
    print("PERHATIAN: ada", info["invalid"].sum(), "baris invalid yang dibuang.")
if info["fs_aktual"].notna().any():
    print("Rentang rata-rata FS file:",
          round(info["fs_aktual"].min(), 1), "s.d.",
          round(info["fs_aktual"].max(), 1), "Hz")
    if ((info["fs_aktual"] - FS).abs() > 0.05 * FS).any():
        print("PERINGATAN: beberapa file berbeda >5% dari FS target; FFT perlu dicermati.")

### Visualisasi A — Berapa banyak file dan seberapa stabil sampling?

Grafik pertama memeriksa keseimbangan kelas; grafik kedua membandingkan **sampling aktual tiap file** dengan target 200 Hz.

In [ ]:
# Dua grafik ringkas untuk inspeksi dataset
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

jumlah = info["label"].value_counts().reindex(KELAS, fill_value=0)
axes[0].bar(jumlah.index, jumlah.values)
axes[0].set_title("Jumlah file per kelas")
axes[0].set_ylabel("File")
axes[0].grid(axis="y", alpha=0.3)

axes[1].plot(range(1, len(info) + 1), info["fs_aktual"], "o")
axes[1].axhline(FS, linestyle="--", label="Target 200 Hz")
axes[1].set_title("Rata-rata sampling setiap file")
axes[1].set_xlabel("Nomor file")
axes[1].set_ylabel("Hz")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Tahap 4 — Visualisasi sinyal getaran mentah

Bandingkan **3 detik pertama** data normal dan anomali pada sumbu X, Y, Z.

Kita ubah angka sensor menjadi satuan `g`, lalu kurangi rata-rata masing-masing sumbu agar getaran lebih terlihat tanpa komponen gravitasi/offset. Ini disebut **menghilangkan komponen DC**.

In [ ]:
# Pilih contoh satu file normal dan satu file anomali
contoh = {}
for kelas in KELAS:
    nama = info.loc[info["label"] == kelas, "file"].iloc[0]
    contoh[kelas] = rekaman[nama]["data"]

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)
for ax, kelas in zip(axes, KELAS):
    df = contoh[kelas].iloc[:3 * FS]
    t = (df["t_us"].to_numpy() - df["t_us"].iloc[0]) / 1e6
    g = df[SUMBU].to_numpy() / SKALA
    g = g - g.mean(axis=0)  # Menghilangkan DC untuk visualisasi
    for i, sumbu in enumerate(["X", "Y", "Z"]):
        ax.plot(t, g[:, i], label=sumbu, linewidth=0.8)
    ax.set_title("Sinyal getaran: " + kelas)
    ax.set_ylabel("Percepatan dinamis (g)")
    ax.legend(loc="upper right", ncol=3)
    ax.grid(alpha=0.3)
axes[-1].set_xlabel("Waktu (detik)")
plt.tight_layout()
plt.show()

## Tahap 5 — Memahami *windowing* (memotong sinyal)

Model tidak langsung menerima satu file berisi ribuan sampel. Kita membagi setiap rekaman menjadi potongan pendek.

**Contoh:** 6.000 sampel dibagi 200 sampel/window = **30 window**. Window tidak saling tumpang tindih (*non-overlapping*). Grafik di bawah menunjukkan tiga window pertama.

In [ ]:
# Ambil sumbu X dari file normal, hanya 3 window pertama
x = contoh["normal"]["ax_raw"].to_numpy()[:3 * WINDOW] / SKALA
x = x - x.mean()
t = np.arange(len(x)) / FS

plt.figure(figsize=(11, 3.5))
plt.plot(t, x, linewidth=0.8)
for k in range(3):
    plt.axvspan(k * WINDOW / FS, (k + 1) * WINDOW / FS,
                alpha=0.12, label="Window " + str(k + 1))
plt.title("Contoh windowing — sumbu X")
plt.xlabel("Waktu perkiraan (detik)")
plt.ylabel("Percepatan dinamis (g)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Tahap 6 — Ekstraksi fitur: mengubah sinyal menjadi angka

Model Random Forest tidak langsung menerima grafik getaran. Untuk setiap potongan **200 sampel (1 window)**, kita menghitung tiga hal pada sumbu X, Y, dan Z:

| Fitur | Makna | Contoh nama kolom |
|---|---|---|
| **RMS** | Seberapa besar getaran yang terjadi | `rms_x`, `rms_y`, `rms_z` |
| **Peak-to-peak** | Selisih amplitudo tertinggi dan terendah | `ptp_x`, `ptp_y`, `ptp_z` |
| **Frekuensi dominan** | Frekuensi dengan puncak FFT terbesar | `freq_dom_x`, `freq_dom_y`, `freq_dom_z` |
| **RMS 3D** | Gabungan besar getaran ketiga sumbu | `rms_3d` |

**Jumlah input model = (3 fitur × 3 sumbu) + 1 fitur gabungan = 10 fitur.**

### Mengapa standar deviasi tidak digunakan lagi?

Dalam program, sinyal terlebih dahulu dikurangi rata-ratanya untuk menghilangkan pengaruh gravitasi/offset (**komponen DC**). Pada sinyal yang sudah memiliki rata-rata nol ini, **RMS dan standar deviasi populasi menghasilkan angka yang sama**. Jadi kita cukup memilih salah satunya: **RMS**. Ini bukan berarti standar deviasi selalu sama dengan RMS pada semua sinyal.

> FFT di sini menggunakan asumsi sampling 200 Hz. Jika sampling sebenarnya tidak stabil, pembacaan frekuensi perlu dicermati.

In [ ]:
# Fungsi ini mengubah 1 window (200 sampel × 3 sumbu) menjadi 10 fitur.

def ekstrak_fitur(window_raw):
    # 1. Konversi data mentah accelerometer menjadi satuan g
    g = window_raw / SKALA

    # 2. Kurangi rata-rata pada X, Y, Z agar yang dihitung terutama getarannya
    g = g - g.mean(axis=0)

    hasil = {}
    for i, sumbu in enumerate(["x", "y", "z"]):
        a = g[:, i]  # Sinyal dari satu sumbu sepanjang 1 window

        # Fitur 1: amplitudo getaran efektif
        hasil[f"rms_{sumbu}"] = np.sqrt(np.mean(a ** 2))

        # Fitur 2: beda nilai terbesar dan terkecil
        hasil[f"ptp_{sumbu}"] = np.ptp(a)

        # Fitur 3: frekuensi pada puncak spektrum FFT
        frekuensi = np.fft.rfftfreq(len(a), d=1 / FS)
        spektrum = np.abs(np.fft.rfft(a))
        spektrum[0] = 0  # Abaikan frekuensi 0 Hz (DC)
        hasil[f"freq_dom_{sumbu}"] = frekuensi[np.argmax(spektrum)]

    # Fitur ke-10: RMS gabungan sumbu X, Y, Z
    hasil["rms_3d"] = np.sqrt(np.mean(np.sum(g ** 2, axis=1)))
    return hasil

# Contoh: tampilkan 10 angka dari window pertama
nama_contoh = info["file"].iloc[0]
satu_window = rekaman[nama_contoh]["data"][SUMBU].iloc[:WINDOW].to_numpy()
print("10 fitur dari satu window:")
display(pd.Series(ekstrak_fitur(satu_window)).round(4).to_frame("nilai"))

### Visualisasi B — FFT: melihat komponen frekuensi

FFT membantu melihat **frekuensi yang menonjol** pada getaran. Grafik ini hanya memakai **window pertama, sumbu X**, dari contoh kedua kelas. Puncak tertinggi tidak selalu sama dengan kerusakan; dalam praktikum ini kita hanya membedakan pola.

In [ ]:
# Bandingkan spektrum FFT sumbu X untuk 1 window per kelas
plt.figure(figsize=(11, 3.5))
for kelas in KELAS:
    a = contoh[kelas]["ax_raw"].to_numpy()[:WINDOW] / SKALA
    a = a - a.mean()
    spektrum = np.abs(np.fft.rfft(a))
    frekuensi = np.fft.rfftfreq(len(a), d=1 / FS)
    plt.plot(frekuensi, spektrum, label=kelas)
plt.title("FFT getaran sumbu X (window pertama)")
plt.xlabel("Frekuensi (Hz)")
plt.ylabel("Magnitudo FFT (relatif)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### Ekstraksi fitur seluruh rekaman

Kita ulangi perhitungan tadi untuk setiap window pada semua file. Hasilnya: **1 window = 1 baris dan 10 angka fitur**.

Kolom `file`, `window`, dan `label` berfungsi sebagai identitas/target dan **tidak masuk ke input model**.

In [ ]:
# Ulangi ekstraksi fitur untuk setiap window setiap file
baris = []
for nama, isi in rekaman.items():
    angka = isi["data"][SUMBU].to_numpy()
    n_window = len(angka) // WINDOW    # Sisa yang < 200 sampel tidak dipakai

    for k in range(n_window):
        awal = k * WINDOW
        potongan = angka[awal:awal + WINDOW]
        fitur_satu = ekstrak_fitur(potongan)
        fitur_satu.update({"file": nama, "label": isi["label"], "window": k + 1})
        baris.append(fitur_satu)

fitur = pd.DataFrame(baris)
KOLOM_FITUR = [c for c in fitur.columns if c not in ["file", "label", "window"]]

print("Ukuran dataset fitur:", fitur.shape)
print("Jumlah fitur untuk model:", len(KOLOM_FITUR))
print("Nama fitur input:", ", ".join(KOLOM_FITUR))
assert len(KOLOM_FITUR) == 10, "Periksa ekstraksi fitur: seharusnya 10 kolom!"
print("Jumlah window per kelas:")
print(fitur["label"].value_counts().to_string())
display(fitur[["file", "window", "label"] + KOLOM_FITUR[:4]].head())

## Tahap 7 — Membagi data: training, validation, dan testing

**Bagian yang sangat penting:** pembagian dilakukan berdasarkan **file CSV**, bukan mengacak semua window. Sebab window yang berasal dari file sama cenderung sangat mirip.

Untuk **15 file per kelas**, kita gunakan:
- **9 file training** — dipakai model untuk belajar.
- **3 file validation** — memantau kinerja sebelum melihat test.
- **3 file testing** — evaluasi akhir pada rekaman yang tidak pernah dilihat model.

Pembagian diacak menggunakan `SEED = 42` supaya dapat diulang.

In [ ]:
# Acak nama FILE untuk tiap kelas, bukan window
rng = np.random.default_rng(SEED)
file_train, file_val, file_test = [], [], []

for kelas in KELAS:
    daftar = info.loc[info["label"] == kelas, "file"].tolist()
    if len(daftar) < 5:
        raise ValueError(f"Kelas {kelas}: minimal 5 file diperlukan untuk 3 subset.")
    rng.shuffle(daftar)

    # Tepat 15 file -> 3 test, 3 validation, 9 training
    n_test = max(1, round(len(daftar) * 0.2))
    n_val = max(1, round(len(daftar) * 0.2))
    file_test.extend(daftar[:n_test])
    file_val.extend(daftar[n_test:n_test + n_val])
    file_train.extend(daftar[n_test + n_val:])

train = fitur[fitur["file"].isin(file_train)].copy()
valid = fitur[fitur["file"].isin(file_val)].copy()
test = fitur[fitur["file"].isin(file_test)].copy()

# Verifikasi bahwa tidak ada file yang masuk ke dua kelompok
assert set(file_train).isdisjoint(file_val)
assert set(file_train).isdisjoint(file_test)
assert set(file_val).isdisjoint(file_test)

print("Jumlah file — train:", len(file_train),
      "| validation:", len(file_val), "| test:", len(file_test))
print("Jumlah window — train:", len(train),
      "| validation:", len(valid), "| test:", len(test))

# Grafik jumlah file per subset dan per kelas
jumlah_split = pd.DataFrame({
    "train": [sum(info.set_index("file").loc[file_train, "label"] == k) for k in KELAS],
    "validation": [sum(info.set_index("file").loc[file_val, "label"] == k) for k in KELAS],
    "test": [sum(info.set_index("file").loc[file_test, "label"] == k) for k in KELAS]
}, index=KELAS)
display(jumlah_split)
jumlah_split.plot(kind="bar", figsize=(7, 3.5), rot=0)
plt.title("Jumlah file tiap subset (bukan jumlah window)")
plt.ylabel("File")
plt.grid(axis="y", alpha=0.3)
plt.show()

### Visualisasi C — Apakah fitur dua kelas berbeda?

Plot di bawah menggunakan **data TRAINING saja** untuk melihat dua fitur: `rms_3d` dan `freq_dom_x`. Warna/posisi yang terpisah bisa membantu klasifikasi, tetapi **tidak menjamin** hasil testing bagus.

In [ ]:
# Dua grafik sederhana berdasarkan fitur TRAINING
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

# Boxplot RMS total 3 sumbu untuk masing-masing kelas
axes[0].boxplot([train.loc[train["label"] == k, "rms_3d"] for k in KELAS],
                tick_labels=KELAS)
axes[0].set_title("Sebaran RMS 3D (train)")
axes[0].set_ylabel("RMS (g)")
axes[0].grid(axis="y", alpha=0.3)

# Scatter plot dua fitur untuk melihat pola
for kelas in KELAS:
    sub = train[train["label"] == kelas]
    axes[1].scatter(sub["rms_3d"], sub["freq_dom_x"],
                    s=15, alpha=0.65, label=kelas)
axes[1].set_title("Dua fitur dari setiap window (train)")
axes[1].set_xlabel("RMS 3D (g)")
axes[1].set_ylabel("Frekuensi dominan X (Hz)")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Tahap 8 — Melatih model Random Forest

Random Forest adalah kumpulan **decision tree**. Setiap pohon mencoba memisahkan `normal` dan `anomali` berdasarkan angka fitur, lalu hasilnya digabung untuk memperoleh prediksi.

**Input (`X`) = 10 kolom fitur** dari tahap sebelumnya.  
**Target (`y`) = kolom `label`** yang berisi `normal` atau `anomali`.

Nama file dan nomor window **tidak digunakan sebagai input model**. Model hanya belajar dari karakteristik getaran.

In [ ]:
# Training hanya memakai dataset train
model = RandomForestClassifier(
    n_estimators=100,       # Jumlah decision tree
    random_state=SEED,
    n_jobs=-1
)
model.fit(train[KOLOM_FITUR], train["label"])

# Validation: mengecek model sebelum pengujian akhir
pred_val = model.predict(valid[KOLOM_FITUR])
print("Akurasi validation:", round(accuracy_score(valid["label"], pred_val) * 100, 2), "%")
print("Catatan: validation bukan hasil akhir; hasil akhir diukur pada TEST.")

## Tahap 9 — Mengevaluasi model pada data testing

- **Accuracy** = persentase prediksi benar.
- **Precision** = dari prediksi suatu kelas, berapa yang benar.
- **Recall** = dari data asli suatu kelas, berapa yang dikenali.
- **F1-score** = keseimbangan precision dan recall.
- **Confusion matrix** menunjukkan jumlah prediksi benar/salah per kelas.

Pengujian ini belum pernah dipakai untuk `model.fit()`.

In [ ]:
# Prediksi pada window dari file TEST yang belum dilihat model
test["prediksi"] = model.predict(test[KOLOM_FITUR])

print("=== HASIL TEST PER WINDOW ===")
print("Akurasi:", round(accuracy_score(test["label"], test["prediksi"]) * 100, 2), "%")
print(classification_report(test["label"], test["prediksi"],
                            labels=KELAS, zero_division=0))

# Visualisasi confusion matrix
ConfusionMatrixDisplay.from_predictions(
    test["label"], test["prediksi"], labels=KELAS,
    cmap="Blues", values_format="d"
)
plt.title("Confusion matrix — TEST per window")
plt.show()

### Evaluasi tambahan — Prediksi per file rekaman

Satu file berisi banyak window. Kita gunakan **voting mayoritas** dari semua window untuk memperoleh **satu prediksi per file**.

Evaluasi ini penting karena unit rekaman independen adalah **file**, bukan window. Dengan hanya **3 file test per kelas**, angka akurasi per file masih mudah berubah; jangan menganggapnya sebagai bukti performa pada semua kompresor.

In [ ]:
# Kumpulkan prediksi window dari file yang sama
per_file = (
    test.groupby("file")
        .agg(label_asli=("label", "first"),
             prediksi=("prediksi", lambda s: s.mode().iloc[0]),
             jumlah_window=("prediksi", "size"))
        .reset_index()
)

print("=== HASIL TEST PER FILE ===")
display(per_file)
print("Akurasi per file:", round(accuracy_score(per_file["label_asli"],
                                              per_file["prediksi"]) * 100, 2), "%")
print(classification_report(per_file["label_asli"], per_file["prediksi"],
                            labels=KELAS, zero_division=0))

### Visualisasi D — Fitur mana yang paling membantu model?

Diagram menunjukkan **feature importance** untuk seluruh **10 fitur**. Semakin tinggi nilainya, semakin sering fitur tersebut membantu pemisahan dalam pohon-pohon Random Forest.

Nilai ini bukan bukti penyebab fisik getaran. Perhatikan juga bahwa beberapa fitur bisa saling berkaitan.

In [ ]:
# Tampilkan pentingnya seluruh 10 fitur menurut Random Forest
penting = pd.Series(model.feature_importances_, index=KOLOM_FITUR)
penting = penting.sort_values(ascending=False)

plt.figure(figsize=(8, 4.2))
penting.sort_values().plot(kind="barh")
plt.title("Pengaruh 10 fitur — Random Forest")
plt.xlabel("Feature importance (relatif)")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## Tahap 10 — Simpan hasil ke laptop

File ZIP hasil berisi:
1. `dataset_fitur.csv` — seluruh fitur per window.
2. `prediksi_window_test.csv` — hasil prediksi pada window testing.
3. `prediksi_file_test.csv` — hasil voting per rekaman testing.
4. `random_forest.joblib` — model yang sudah dilatih dan nama fitur untuk digunakan lagi.

Klik tautan unduhan yang otomatis muncul setelah menjalankan cell.

In [ ]:
# Simpan dataset fitur dan model ke folder hasil
folder_hasil = Path("hasil_klasifikasi")
folder_hasil.mkdir(exist_ok=True)

fitur.to_csv(folder_hasil / "dataset_fitur.csv", index=False)
test.to_csv(folder_hasil / "prediksi_window_test.csv", index=False)
per_file.to_csv(folder_hasil / "prediksi_file_test.csv", index=False)
joblib.dump({"model": model, "features": KOLOM_FITUR,
             "fs": FS, "window": WINDOW, "skala": SKALA},
            folder_hasil / "random_forest.joblib")

# ZIP hasil agar mahasiswa mengunduh satu file saja
nama_hasil_zip = shutil.make_archive("hasil_klasifikasi_kompresor", "zip", folder_hasil)
print("File hasil siap diunduh:", nama_hasil_zip)
files.download(nama_hasil_zip)

## Latihan mahasiswa

1. **Analisis grafik:** pada grafik getaran dan FFT, apa perbedaan yang terlihat antara kelas `normal` dan `anomali`? Apakah perbedaannya selalu berupa amplitudo yang lebih tinggi?
2. **Eksperimen panjang window:** ubah `WINDOW = 200` menjadi `WINDOW = 100` (sekitar 0,5 detik), lalu jalankan kembali mulai Tahap 5. Bagaimana perubahan jumlah window, resolusi frekuensi FFT, dan akurasi testing?
3. **Analisis fitur:** menurut grafik *feature importance*, fitur mana paling berpengaruh dari 10 fitur? Apakah hasil pada sumbu X, Y, atau Z mendominasi?

**Kesimpulan penting:** Praktikum ini mengklasifikasikan **perbedaan pola getaran yang direkam**, bukan mendiagnosis kerusakan internal kompresor. Model juga belum tentu berlaku jika posisi sensor, kondisi operasi, atau jenis kompresor berubah.